In [1]:
import torch
import torch.nn as nn
import pandas as pd

print("PyTorch version:", torch.__version__)
print("Setup successful!")

PyTorch version: 2.14.1+cpu
Setup successful!


In [2]:
class Standard_CNN(nn.Module):
    def __init__(self):#modified vgg16+lenet5
        super().__init__()
        self.Conv1 = nn.Conv2d(1,6,kernel_size=3,stride=1,padding=1)#maxpooling doesnt change the cin or cout.. only downsample the picture
        self.batch_norm1 = nn.BatchNorm2d(6)
        self.Conv2 = nn.Conv2d(6,16,kernel_size=3,stride=1,padding=1)
        self.batch_norm2 = nn.BatchNorm2d(16)
        self.fc1    = nn.Linear(16*4*4,120)
        self.fc2    = nn.Linear(120,84)
        self.fc3    = nn.Linear(84,10)
    def forward(self,x):
        x = torch.relu(self.batch_norm1(self.Conv1(x)))
        x = torch.max_pool2d(x,kernel_size=2,stride=2)
        x = torch.relu(self.batch_norm2(self.Conv2(x)))
        x = torch.max_pool2d(x,kernel_size=2,stride=2)
        x = torch.max_pool2d(x,kernel_size=2,stride=2)
        x = x.view(-1,16*4*4)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [3]:
class DepthWise_Separable_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1_depth = nn.Conv2d(1,1,kernel_size=3,padding=1,groups=1)
        self.batch_norm_1_1 = nn.BatchNorm2d(1)
        self.conv1_point = nn.Conv2d(1,6,kernel_size=1,groups=1)
        self.batch_norm_1_2 = nn.BatchNorm2d(6)
        self.conv2_depth = nn.Conv2d(6,6,kernel_size=3,padding=1,groups=6)
        self.batch_norm_2_1 = nn.BatchNorm2d(6)
        self.conv2_point = nn.Conv2d(6,16,kernel_size=1,groups=1)
        self.batch_norm_2_2 = nn.BatchNorm2d(16)
        self.fc1    = nn.Linear(16*4*4,120)
        self.fc2    = nn.Linear(120,84)
        self.fc3    = nn.Linear(84,10)

    def forward(self,x):
        x = self.conv1_depth(x)
        x = self.batch_norm_1_1(x)
        x = torch.relu(x)
        x = self.conv1_point(x)
        x = self.batch_norm_1_2(x)
        x = torch.relu(x)
        x = torch.max_pool2d(x,kernel_size=2,stride=2)
        x = self.conv2_depth(x)
        x = self.batch_norm_2_1(x)
        x = torch.relu(x)
        x = self.conv2_point(x)
        x = self.batch_norm_2_2(x)
        x = torch.relu(x)
        x = torch.max_pool2d(x,kernel_size=2,stride=2)
        x = torch.max_pool2d(x,kernel_size=2,stride=2)
        x = x.view(-1,16*4*4)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [4]:
model_A = Standard_CNN()
model_B = DepthWise_Separable_CNN()

print("Both CNN models created successfully!")

Both CNN models created successfully!


In [5]:
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


params_A = count_parameters(model_A)
params_B = count_parameters(model_B)

print("Model A trainable parameters:", params_A)
print("Model B trainable parameters:", params_B)

if params_B <= 100000:
    print("Model B satisfies the 100,000 parameter limit.")
else:
    print("Model B exceeds the parameter limit!")

Model A trainable parameters: 42838
Model B trainable parameters: 42106
Model B satisfies the 100,000 parameter limit.


In [6]:
def estimated_weight_size_kb(model):
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    return total_bytes / 1024


size_A = estimated_weight_size_kb(model_A)
size_B = estimated_weight_size_kb(model_B)

print(f"Model A weight size: {size_A:.2f} KB")
print(f"Model B weight size: {size_B:.2f} KB")

Model A weight size: 167.34 KB
Model B weight size: 164.48 KB


In [7]:
comparison = pd.DataFrame({
    "Model": [
        "Standard CNN",
        "Depthwise Separable CNN"
    ],
    "Trainable Parameters": [
        params_A,
        params_B
    ],
    "Estimated Weight Size (KB)": [
        round(size_A, 2),
        round(size_B, 2)
    ]
})

display(comparison)

,Model,Trainable Parameters,Estimated Weight Size (KB)
0,Standard CNN,42838,167.34
1,Depthwise Separable CNN,42106,164.48


In [8]:
reduction = (params_A - params_B) / params_A * 100

print(f"Parameter reduction: {reduction:.2f}%")

Parameter reduction: 1.71%
